In [1]:
import csv
import re

In [2]:
records  = []
patterns = []

In [3]:
PIPELINES_FROM_REDMINES  = "save-issues\\redmine_issues.csv"
PIPELINES_PROCESSED      = "pipelines_with_appcodes.csv"
PIPELINES_TO_BE_IGNORED  = "patterns.txt"

In [4]:
# Function to load regex patterns from a file
def load_ignore_patterns(file_path):
    patterns = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                patterns.append(line)
    # Combine patterns into a single regex
    if patterns:
        return re.compile("|".join(patterns))
    return None

# Load regex patterns
ignore_regex = load_ignore_patterns(PIPELINES_TO_BE_IGNORED)

In [5]:
total_pipelines = 0
match_pipelines = 0

with open(PIPELINES_FROM_REDMINES, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for record in reader:
        #
        # Save the URL of all pipelines
        #
        develop_pipeline  = record.get("custom_URL_Job_Jenkins_Develop", "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        master_pipeline   = record.get("custom_URL_Job_Jenkins_Master",  "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        release_pipeline  = record.get("custom_URL_Job_Jenkins_Release", "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        it_pipeline       = record.get("custom_URL_Job_Jenkins_IT",      "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        lab_pipeline      = record.get("custom_URL_Job_Jenkins_Lab",     "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")
        main_pipeline     = record.get("custom_URL_Job_Jenkins_Main",    "").replace("http://172.16.63.239/jenkins/job/", "").replace("job/", "")

        #
        # Save the application code
        # 
        match = re.search(r"[A-Z]\d+", record.get("custom_Codice_Applicativo", ""))
        code  = match.group(0) if match else ""

        pipelines = [ develop_pipeline, master_pipeline, release_pipeline, it_pipeline, lab_pipeline, main_pipeline ]
        for p in pipelines:
            total_pipelines += 1
            if p and not (ignore_regex and ignore_regex.search(p)):
                match_pipelines += 1
                group_name = p.split("/")[0].strip()
                pipe_name  = p.split("/")[1].strip()
                record = {"group_name": group_name, "pipeline_name": pipe_name, "application_code": code}
                records.append(record)

print(f"Total pipelines:    {total_pipelines}")
print(f"Matching pipelines: {match_pipelines}")
print(f"Ignored pipelines:  {total_pipelines - match_pipelines}")

Total pipelines:    25014
Matching pipelines: 8850
Ignored pipelines:  16164


In [6]:
with open(PIPELINES_PROCESSED, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["group_name", "pipeline_name", "application_code"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)